In [1]:
BANNER = \
"""
# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [2]:
# ------------------------------------------------------------
# container runtime
# ------------------------------------------------------------

# note:
# this notebook assumes that notebook 01 completed successfully
# using the container runtime configured in 00_penguins_db_boot.py
#
# recall that 00_penguins_db_boot.py is your working copy of
# _mLDm_00_penguins_db_boot.py
#
# exactly one of these alternatives must be active:
#
# RUNTIME = "podman"
# # RUNTIME = "docker"
#
# or:
#
# RUNTIME = "podman"
RUNTIME = "docker"

In [3]:
# ------------------------------------------------------------
# connect DuckDB to the integrated PostgreSQL data source
# ------------------------------------------------------------

import duckdb

# create an in-memory DuckDB connection
db = duckdb.connect()

# install and load the DuckDB PostgreSQL extension
# the INSTALL statement is required only the first time the extension is used
db.sql("""
INSTALL postgres;
LOAD postgres;
""")

# attach the PostgreSQL database to DuckDB
# the database will be accessible through the name "pg"
db.sql("""
ATTACH
'host=localhost port=5432 dbname=mldm user=mldm password=mldm'
AS pg
(TYPE postgres);
""")

print("PTS | DuckDB connected to PostgreSQL")

PTS | DuckDB connected to PostgreSQL


In [4]:
# ------------------------------------------------------------
# inspect observations with missing island information
# ------------------------------------------------------------

# check whether geographic coordinates are available when island is missing
db.sql("""
SELECT
    id_penguin,
    date_time,
    latitude,
    longitude
FROM pg.public.observation
WHERE name_island IS NULL
ORDER BY id_penguin;
""").show()

# expected result:
# 20 observations

┌────────────┬─────────────────────┬────────────────┬────────────────┐
│ id_penguin │      date_time      │    latitude    │   longitude    │
│   int32    │      timestamp      │     double     │     double     │
├────────────┼─────────────────────┼────────────────┼────────────────┤
│          9 │ 2007-11-10 08:00:00 │ -65.4438803787 │ -65.4708759261 │
│         15 │ 2007-11-12 08:30:00 │ -65.4524372344 │ -65.4992878878 │
│         21 │ 2007-11-12 16:00:00 │  -65.440375842 │ -65.6168380872 │
│         22 │ 2007-11-13 08:30:00 │ -64.6618522499 │ -64.1795350838 │
│         25 │ 2007-11-13 11:30:00 │ -64.7416340146 │ -64.3250463705 │
│         28 │ 2007-11-15 10:30:00 │ -64.7723988048 │ -64.1498490806 │
│         53 │ 2007-11-19 16:00:00 │ -64.7452407798 │ -64.2772332674 │
│        136 │ 2008-11-05 09:30:00 │ -64.7535363708 │ -64.2977509357 │
│        145 │ 2008-11-06 14:00:00 │ -65.4546283597 │  -65.556591072 │
│        154 │ 2008-11-08 11:00:00 │ -64.7377056137 │ -64.1783600441 │
│     

In [5]:
# ------------------------------------------------------------
# calculate distances from observations to island references
#
# note:
# create the view in PostgreSQL so that the provided
# haversine_km function can be used directly
# ------------------------------------------------------------

db.sql("""
CALL postgres_execute(
    'pg',
    '
    CREATE OR REPLACE VIEW observation_island_distances AS
    SELECT
        o.id_penguin,
        o.date_time,
        i.name AS island,
        haversine_km(
            o.latitude,
            o.longitude,
            i.centroid_latitude,
            i.centroid_longitude
        ) AS distance_km
    FROM observation AS o
    CROSS JOIN island AS i
    WHERE o.name_island IS NULL
    '
);
""")

# inspect distances from observations to islands
db.sql("""
SELECT *
FROM pg.public.observation_island_distances
ORDER BY id_penguin, date_time, distance_km;
""").show()

# expected result:
# 60 distances = 20 observations × 3 island references

┌────────────┬─────────────────────┬───────────┬────────────────────┐
│ id_penguin │      date_time      │  island   │    distance_km     │
│   int32    │      timestamp      │  varchar  │       double       │
├────────────┼─────────────────────┼───────────┼────────────────────┤
│          9 │ 2007-11-10 08:00:00 │ Biscoe    │  2.048035593933077 │
│          9 │ 2007-11-10 08:00:00 │ Dream     │  98.38026400404114 │
│          9 │ 2007-11-10 08:00:00 │ Torgersen │  99.25681464946348 │
│         15 │ 2007-11-12 08:30:00 │ Biscoe    │ 2.4951271348048194 │
│         15 │ 2007-11-12 08:30:00 │ Dream     │  99.92956542473108 │
│         15 │ 2007-11-12 08:30:00 │ Torgersen │ 100.84066526163899 │
│         21 │ 2007-11-12 16:00:00 │ Biscoe    │  5.522843595478677 │
│         21 │ 2007-11-12 16:00:00 │ Dream     │ 102.26828341632346 │
│         21 │ 2007-11-12 16:00:00 │ Torgersen │ 103.58704542045122 │
│         22 │ 2007-11-13 08:30:00 │ Dream     │ 7.9482102415582165 │
│          · │      

In [6]:
# ------------------------------------------------------------
# obtain the minimum island distance for each observation
# ------------------------------------------------------------

db.sql("""
CREATE OR REPLACE TEMP VIEW observation_min_distance AS
SELECT id_penguin,
       date_time,
       MIN(distance_km) AS distance_km
FROM pg.public.observation_island_distances
GROUP BY id_penguin, date_time;
""")

# inspect minimum distances from observations to islands
db.sql("""
SELECT *
FROM observation_min_distance
ORDER BY distance_km;
""").show()

# expected result:
# 20 rows — one minimum distance per observation

┌────────────┬─────────────────────┬────────────────────┐
│ id_penguin │      date_time      │    distance_km     │
│   int32    │      timestamp      │       double       │
├────────────┼─────────────────────┼────────────────────┤
│          9 │ 2007-11-10 08:00:00 │  2.048035593933077 │
│        173 │ 2008-11-09 15:30:00 │  2.053646939928896 │
│         15 │ 2007-11-12 08:30:00 │ 2.4951271348048194 │
│        154 │ 2008-11-08 11:00:00 │ 2.5963250291332067 │
│        205 │ 2008-11-17 10:30:00 │  2.611115445553207 │
│         53 │ 2007-11-19 16:00:00 │ 2.8099696876669142 │
│         28 │ 2007-11-15 10:30:00 │   3.32123450367243 │
│        145 │ 2008-11-06 14:00:00 │ 3.7867361563129296 │
│        269 │ 2009-11-18 10:30:00 │  4.045318446653998 │
│        180 │ 2008-11-11 14:30:00 │  4.122264704134482 │
│        136 │ 2008-11-05 09:30:00 │  4.145212344602117 │
│         25 │ 2007-11-13 11:30:00 │ 4.6924802842640325 │
│        155 │ 2008-11-08 12:00:00 │  5.365421463423447 │
│         21 │

In [7]:
# ------------------------------------------------------------
# identify the nearest island for each observation
# ------------------------------------------------------------

db.sql("""
CREATE OR REPLACE TEMP VIEW observation_nearest_island AS
SELECT d.id_penguin,
       d.date_time,
       d.island,
       d.distance_km
FROM pg.public.observation_island_distances AS d JOIN observation_min_distance AS m
     ON d.id_penguin == m.id_penguin and d.date_time == m.date_time
     AND d.distance_km <= m.distance_km;
""")

db.sql("""
SELECT *
FROM observation_nearest_island
ORDER BY id_penguin;
""").show()

# expected result:
# 20 rows — one nearest island per observation

┌────────────┬─────────────────────┬───────────┬────────────────────┐
│ id_penguin │      date_time      │  island   │    distance_km     │
│   int32    │      timestamp      │  varchar  │       double       │
├────────────┼─────────────────────┼───────────┼────────────────────┤
│          9 │ 2007-11-10 08:00:00 │ Biscoe    │  2.048035593933077 │
│         15 │ 2007-11-12 08:30:00 │ Biscoe    │ 2.4951271348048194 │
│         21 │ 2007-11-12 16:00:00 │ Biscoe    │  5.522843595478677 │
│         22 │ 2007-11-13 08:30:00 │ Dream     │ 7.9482102415582165 │
│         25 │ 2007-11-13 11:30:00 │ Dream     │ 4.6924802842640325 │
│         28 │ 2007-11-15 10:30:00 │ Torgersen │   3.32123450367243 │
│         53 │ 2007-11-19 16:00:00 │ Dream     │ 2.8099696876669142 │
│        136 │ 2008-11-05 09:30:00 │ Dream     │  4.145212344602117 │
│        145 │ 2008-11-06 14:00:00 │ Biscoe    │ 3.7867361563129296 │
│        154 │ 2008-11-08 11:00:00 │ Dream     │ 2.5963250291332067 │
│        155 │ 2008-

In [8]:
# ------------------------------------------------------------
# update observations with the inferred island
# ------------------------------------------------------------

# assign the nearest island to observations with missing island information
db.sql("""
UPDATE pg.public.observation AS o
       SET name_island = n.island
FROM observation_nearest_island AS n
WHERE o.id_penguin = n.id_penguin
  AND o.date_time = n.date_time
  AND o.name_island IS NULL;
""")

# verify that all observations now have an island
db.sql("""
SELECT COUNT(*) AS n_missing_island
FROM pg.public.observation
WHERE name_island IS NULL;
""").show()

# expected result:
# n_missing_island = 0

┌──────────────────┐
│ n_missing_island │
│      int64       │
├──────────────────┤
│                0 │
└──────────────────┘



In [9]:
# ------------------------------------------------------------
# remove the auxiliary view used during source completion
# ------------------------------------------------------------

# remove the postgresql view so the database is left without completion-specific objects
db.sql("""
CALL postgres_execute(
    'pg',
    'DROP VIEW IF EXISTS observation_island_distances'
);
""")

┌─────────┐
│ Success │
│ boolean │
└─────────┘
  0 rows 

# Visual checkpoint with Orange Data Mining (DM)

*Note:* Before completing the **final evidence question** (cf. next cell), take a first visual look at the collected data using **Orange DM**.

## 1. Install Orange DM

Orange DM is used as a **standalone application** and should **NOT** be installed in the course Python virtual environment (`__venv`).

If Orange DM is not already installed:

**1.a. Open the Orange DM download page**

`https://orangedatamining.com/download/`

**1.b. Select the installer for your operating system**

- **Windows** — download and run the Windows installer
- **macOS (Apple silicon)** — use the Apple silicon version
- **macOS (Intel)** — use the Intel version

**1.c. Complete the installation**

Start **Orange Data Mining** after the installation finishes.

## 2. Small visual exploration

**2.a. Load the collected data**

Add a `File` widget and load:

`01_input/_mLDm_00_penguins_data_collected.csv`

**2.b. Select the attributes**

Add a `Select Columns` widget and connect `File` to `Select Columns`.

In `Select Columns`:

- keep `species` as the **Target**
- choose a subset of attributes to keep as **Features**
- move attributes that you do not want to use directly in this first analysis to **Meta Attributes** or **Ignored**

Do not try to find the “best” feature set at this stage.

The goal is to notice that an attribute may exist in the source but still be excluded from, or assigned a different role in, an analytical view of the data.

## 3. Explore the data visually

Connect the output of `Select Columns` to:

- `Data Table`
- `Distributions`
- `Scatter Plot`

Then:

- use `Data Table` to inspect the resulting data
- use `Distributions` to inspect at least one numerical attribute
- use `Scatter Plot` to explore the relationship between two numerical attributes, using `species` to distinguish the penguins

## 4. Evidence

Save the Orange workflow as:

`02_code/penguins_visual_check.ows`

Save evidence of the Orange exploration in `04_report`.

The evidence must show:

- the workflow and widgets used
- the role assigned to `species`
- the selected and excluded attributes
- at least one visual result

You may use two screenshots, for example:

`04_report/penguins_orange_columns.png`

`04_report/penguins_orange_visualisation.png`

In [10]:
# ------------------------------------------------------------
# final evidence question
# ------------------------------------------------------------

# using evidence from the execution of notebooks 01 and 02, explain:
#
# - why the number of observations differs from the number of distinct penguins
# - how the completion process inferred the missing island information
# - what evidence confirms that the source was completed without changing its
#   observation-level structure
#
# answer in approximately 4–6 sentences
#
# A:
# - After integrating the CSV data source, it can be seen that there are 10 more observations than the total number of unique penguins. Therefore, it can be inferred that a penguin can be observed more than once to gather more specific data about it.
# - Cells 5 to 8 from the completion notebook have imputed the missing island information, by inferring the closest island based on the Haversine distance between an observation's geographic coordinates and the center coordinates of an island.
# - Since the number of total observations and unique penguins remain the same, and each observation is still identified by its penguin ID and timestamp, we can infer that the observation structure was preserved. There was no deleted or duplicated observations; the only change was the imputation of the island name. Another indicator is the final number of observations; after preprocessing, the CSV dataset was fully integrated into the database, bringing the total observation count from 20 to 344.